In [1]:
from pyspark import SparkConf, SparkContext
conf = SparkConf().setAppName("Prova_esame")
sc = SparkContext(conf=conf)

#Task 1

In [3]:
outputPath1="./output1"
outOfOrdersRDD=sc.textFile("./data/OutOfOrders.txt")
prodPlantsRDD=sc.textFile("./data/ProdPlants.txt")
robotsRDD=sc.textFile("./data/Robots.txt")

In [ ]:
cleanedBrokenRDD=outOfOrdersRDD.map(lambda x: (x.split(",")[0],x.split(",")[1])).filter(lambda x: "2020/01/01"<x[1]<"2021/12/31").distinct()\
    .map(lambda x: (x[0],(1,0)) if x[1].split("/")[0]=='2020' else (x[0],(0,1))).reduceByKey(lambda a,b: (a[0]+b[0],a[1]+b[1])) \
    .filter(lambda x: x[1][0]<x[1][1])

cleanedPlantsRDD=prodPlantsRDD.map(lambda x: (x.split(",")[0],x.split(",")[1]))
cleanedRobotsRDD=robotsRDD.map(lambda x: (x.split(",")[1],x.split(",")[0]))

joinedRDD=cleanedRobotsRDD.join(cleanedPlantsRDD)
joined2RDD=joinedRDD.map(lambda x: (x[1][0],x[1][1])).join(cleanedBrokenRDD).map(lambda x: (x[0],x[1][0]))

#Task 2

In [ ]:
from datetime import datetime, timedelta

# Predefined function
def previousDate(mydate, n):
    currentDate=datetime.strptime(mydate,"%Y/%m/%d")
    prevDate=currentDate-timedelta(days=n)
    return prevDate.strftime("%Y/%m/%d")

In [ ]:
newCleanedBrokenRDD=outOfOrdersRDD.map(lambda x: (x.split(",")[0],x.split(",")[1]))
cleanedRobotsRDD2=robotsRDD.map(lambda x: (x.split(",")[0],x.split(",")[1]))
ridSDatesPlantsRDD=newCleanedBrokenRDD.join(cleanedRobotsRDD2)

def windowElements(tripleRidDatePID):
    RID = tripleRidDatePID[0]
    date = tripleRidDatePID[1][0]
    PID = tripleRidDatePID[1][1]

    elements = []

    elements.append( ((PID, date), RID) )
    elements.append( ((PID, previousDate(date, 1)), RID) )
    elements.append( ((PID, previousDate(date, 2)), RID) )

    return elements

windowsElementsRDD = ridSDatesPlantsRDD.flatMap(windowElements).distinct()

In [ ]:
ridSDatesPlantsRDD.collect()

[('R2', ('2020/05/01', 'PID1')),
 ('R2', ('2020/05/02', 'PID1')),
 ('R2', ('2021/05/06', 'PID1')),
 ('R2', ('2021/05/07', 'PID1')),
 ('R2', ('2021/05/09', 'PID1')),
 ('R2', ('2021/05/08', 'PID1')),
 ('R2', ('2010/05/03', 'PID1')),
 ('R2', ('2010/05/04', 'PID1')),
 ('R2', ('2021/05/09', 'PID1')),
 ('R3', ('2020/05/04', 'PID1')),
 ('R3', ('2021/05/04', 'PID1')),
 ('R3', ('2010/05/04', 'PID1')),
 ('R3', ('2010/05/05', 'PID1')),
 ('R1', ('2020/06/01', 'PID1')),
 ('R1', ('2020/06/02', 'PID1')),
 ('R1', ('2020/06/04', 'PID1')),
 ('R1', ('2020/06/05', 'PID1')),
 ('R1', ('2010/05/02', 'PID1')),
 ('R4', ('2021/05/04', 'PID2')),
 ('R4', ('2010/05/03', 'PID2')),
 ('R5', ('2010/05/01', 'PID2')),
 ('R5', ('2010/05/04', 'PID2'))]